In [1]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "https://mlops-vm.tailc0798c.ts.net:5000"

### Interacting with the MLflow tracking server

The MlflowClient object allows us to interact with...

+ an MLflow Tracking Server that creates and manages experiments and runs.
+ an MLflow Registry Server that creates and manages registered models and model versions.
To instantiate it we need to pass a tracking URI and/or a registry URI

In [70]:
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

client.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/5', creation_time=1786731794129, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1786731794129, lifecycle_stage='active', name='nyc-taxi', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1786723449402, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1786723449402, lifecycle_stage='active', name='new-experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='gs://$GCP_BUCKET/0', creation_time=1786645869124, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1786645869124, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

In [6]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='5',
    filter_string="metrics.rmse < 7 AND tags.estimator_name != ''",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=10,
    order_by=["metrics.rmse ASC"]
)

for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: 20a1e974ff4a46be8810a971026a7089, rmse: 5.4400
run id: e594e03acd5945d2a797e050319c15b8, rmse: 5.4757
run id: f1be7867870c485d8f87a131852d482c, rmse: 5.7446


### Interacting with the Model Registry

In this section We will use the MlflowClient instance to:
+ Register a new model for the experiment nyc-taxi-regressor
+ Retrieve the latests versions of the model nyc-taxi-regressor and check that new versions were created.
+ Set `champion` and `challenger` versions of the model

In [7]:
run_id = "20a1e974ff4a46be8810a971026a7089"
experiment_id = "5"

model_list = client.search_logged_models(
    experiment_ids=[experiment_id],
    filter_string=f"source_run_id = '{run_id}'",
    max_results=1000,
)

for model in model_list:
    print(model)

LoggedModel(artifact_location='mlflow-artifacts:/5/models/m-1a50bd01cd764e578ba7eaa12840fcb4/artifacts', creation_timestamp=1786732641666, experiment_id='5', last_updated_timestamp=1786732653628, model_id='m-1a50bd01cd764e578ba7eaa12840fcb4', model_type='', model_uri='models:/m-1a50bd01cd764e578ba7eaa12840fcb4', name='model', source_run_id='20a1e974ff4a46be8810a971026a7089', status=<LoggedModelStatus.READY: 'READY'>, status_message='')


In [ ]:
registered_model_name="nyc-taxi-regressor"
registered_model = client.create_registered_model(name=registered_model_name)

In [21]:
logged_model_id = "m-1a50bd01cd764e578ba7eaa12840fcb4"
logged_model = client.get_logged_model(selected_model_id)

champion_model = client.create_model_version(
    name=registered_model.name,
    source=logged_model.model_uri,
    run_id=logged_model.source_run_id
)
client.set_registered_model_alias(
    name=champion_model.name,
    alias="champion",
    version=champion_model.version,
)

challenger_model = client.create_model_version(
    name=registered_model.name,
    source=logged_model.model_uri,
    run_id=logged_model.source_run_id
)
client.set_registered_model_alias(
    name=challenger_model.name,
    alias="challenger",
    version=challenger_model.version,
)

2026/08/14 15:32:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: nyc-taxi-regressor, version 1
2026/08/14 15:32:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: nyc-taxi-regressor, version 2


In [22]:
def print_model_info(name):

    models = client.search_model_versions(
        filter_string=f"name = '{name}'",
        order_by=["version_number ASC"],
    )
    
    print(f"Model name: '{name}'")
    
    for model in models:
        version_info = client.get_model_version(
            name=model.name,
            version=model.version,
        )
    
        print(
            f"version={version_info.version}, "
            f"aliases={version_info.aliases}, "
            f"source={version_info.source}"
        )

In [23]:
print_model_info(registered_model.name)

Model name: 'nyc-taxi-regressor'
version=1, aliases=['champion'], source=models:/m-1a50bd01cd764e578ba7eaa12840fcb4
version=2, aliases=['challenger'], source=models:/m-1a50bd01cd764e578ba7eaa12840fcb4


### Comparing versions and selecting the new champion model

The idea is to simulate the scenario in which a deployment engineer has to interact with the model registry to decide whether to update the model version that is in production or not.

These are the steps:
+ Load the test dataset, which corresponds to the NYC Green Taxi data from the month of March 2023.
+ Download the DictVectorizer that was fitted using the training data and saved to MLflow as an artifact, and load it with pickle.
+ Preprocess the test set using the DictVectorizer so we can properly feed the regressors.
+ Make predictions on the test set using the model versions that are currently labeled as `challnger` and `champion`, and compare their performance.
+ Based on the results, set the new `champion` model version accordingly.

In [58]:
from sklearn.metrics import root_mean_squared_error
import pandas as pd
import pickle


def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(model_uri, X_test, y_test):
    model = mlflow.pyfunc.load_model(model_uri)
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}

In [25]:
df = read_dataframe("data/green_tripdata_2023-03.parquet")

In [ ]:
import pickle
import mlflow

mlflow.set_tracking_uri("https://mlops-vm.tailc0798c.ts.net:5000")
mlflow.set_experiment("nyc-taxi")

registered_model_name="nyc-taxi-regressor"

model_version = client.get_model_version_by_alias(
    name=registered_model_name,
    alias="challenger",
)

preprocessor_path = client.download_artifacts(run_id=model_version.run_id, path='preprocessor/preprocessor.b')

with open(preprocessor_path, 'rb') as f_in:
    dv = pickle.load(f_in)

X_test = preprocess(df, dv)

target = "duration"
y_test = df[target].values

%time test_model(model_uri=model_version.source, X_test=X_test, y_test=y_test)

In [ ]:
import pickle
import mlflow

mlflow.set_tracking_uri("https://mlops-vm.tailc0798c.ts.net:5000")
mlflow.set_experiment("nyc-taxi")

registered_model_name="nyc-taxi-regressor"

model_version = client.get_model_version_by_alias(
    name=registered_model_name,
    alias="champion",
)

preprocessor_path = client.download_artifacts(run_id=model_version.run_id, path='preprocessor/preprocessor.b')

with open(preprocessor_path, 'rb') as f_in:
    dv = pickle.load(f_in)

X_test = preprocess(df, dv)

target = "duration"
y_test = df[target].values

%time test_model(model_uri=model_version.source, X_test=X_test, y_test=y_test)

In [69]:
registered_model_name="nyc-taxi-regressor"

client.set_registered_model_alias(
    name=registered_model_name,
    alias="champion",
    version=challenger_version_info.version,
)

client.delete_registered_model_alias(
    name=registered_model_name,
    alias="challenger",
)

print_model_info(registered_model_name)

Model name: 'nyc-taxi-regressor'
version=1, aliases=[], source=models:/m-1a50bd01cd764e578ba7eaa12840fcb4
version=2, aliases=['champion'], source=models:/m-1a50bd01cd764e578ba7eaa12840fcb4
